In [0]:
%pip install confluent_kafka


In [0]:
%restart_python

In [0]:
import json
import os
import time
import random
from datetime import datetime
from confluent_kafka import Producer

# 🔐 Load Confluent credentials from environment variables
bootstrap_servers = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "pkc-xxxxx.us-east1.gcp.confluent.cloud:9092")
username = os.getenv("KAFKA_API_KEY")
password = os.getenv("KAFKA_API_SECRET")

if not username or not password:
    raise ValueError("Missing KAFKA_API_KEY or KAFKA_API_SECRET environment variables")

conf = {
    "bootstrap.servers": bootstrap_servers,
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": username,
    "sasl.password": password,
}

producer = Producer(conf)
topic = os.getenv("KAFKA_TOPIC", "user_events")

# 🛍️ Real-world data lists
event_types = ["view_item", "view_item", "add_to_cart", "click_ad", "purchase"]
item_ids = [f"PROD_{i}" for i in range(101, 150)]
sources = ["Instagram", "Google_Ads", "TikTok", "Direct", "Email_Campaign"]

print("🚀 Starting User Simulator... Streaming live events to Confluent.")

try:
    while True:
        # Create a realistic "Clickstream" event
        data = {
            "event_id": f"evt_{int(time.time() * 1000)}_{random.randint(1000, 9999)}",
            "schema_version": "1.0",
            "user_id": random.randint(5000, 9999),
            "event_type": random.choice(event_types),
            "item_id": random.choice(item_ids),
            "price": round(random.uniform(10.0, 500.0), 2),
            "traffic_source": random.choice(sources),
            "ip_address": f"192.168.{random.randint(0, 255)}.{random.randint(1, 254)}",
            "event_timestamp": datetime.utcnow().isoformat(),
            "ingest_ts": datetime.utcnow().isoformat(),
        }

        producer.produce(topic, json.dumps(data).encode("utf-8"))
        producer.flush()
        print("✅ Sent:", data)

        # Send an event every 1 second
        time.sleep(1)

except KeyboardInterrupt:
    print("
🛑 Simulation stopped.")

